# Purch Orders Search

In [ ]:
import pandas as pd
from IPython.display import display

path = "data/PurPurchOrdersSearch_new.csv"

df = pd.read_csv(
    path,
    sep="\t",
    encoding="utf-16"
)

# remove rows where Item is NaN
df = df.dropna(subset=['Item'])

# convert Receipt date to datetime
df['Receipt date'] = pd.to_datetime(df['Receipt date'], dayfirst=True, errors='coerce')

# delete rows with Receipt date before today
tomorrow = pd.Timestamp.today().normalize() + pd.Timedelta(days=1)
df = df[df['Receipt date'] >= tomorrow]

# sort by SKU (Item) and order number
df = df.sort_values(by=['Item', 'Order number'])

# reset index
df = df.reset_index(drop=True)

display(df.head(100))
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

# export to Excel
output_path = "export/clean_PO.xlsx"
df.to_excel(output_path, index=False)

print("Excel exported:", output_path)

,Header,Currency,Discount,Item,Order date,Order number,Quantity,Unit price,VAT code,Warehouse codeDescription,Receipt date
0,NaN,IDR,NaN,BA027W,11-11-2025,3195,"50,00","35.000,00",G0,01,2026-03-27
1,NaN,IDR,NaN,BA161N,04-11-2025,3181,"300,00","30.000,00",G0,01,2026-04-06
2,NaN,IDR,NaN,BA172N-M-50x50,15-12-2025,3244,"4,00","140.000,00",G0,01,2026-05-01
3,NaN,IDR,NaN,BA172N-M-50x50,15-12-2025,3244,"4,00","140.000,00",G0,01,2026-05-15
4,NaN,IDR,NaN,BA172N-M-50x50,15-12-2025,3244,"12,00","140.000,00",G0,01,2026-05-31
...,...,...,...,...,...,...,...,...,...,...,...
95,NaN,IDR,NaN,BAGE013NBr-L,05-11-2025,3185,"18,00","185.000,00",G0,01,2026-05-01
96,NaN,IDR,NaN,BAGE013NBr-L,05-11-2025,3185,"17,00","185.000,00",G0,01,2026-05-15
97,NaN,IDR,NaN,BAGE013NBr-M,05-11-2025,3185,"40,00","153.000,00",G0,01,2026-04-06
98,NaN,IDR,NaN,BAGE013NBr-M,05-11-2025,3185,"3,00","153.000,00",G0,01,2026-05-01


Rows: 902
Columns: ['Header', 'Currency', 'Discount', 'Item', 'Order date', 'Order number', 'Quantity', 'Unit price', 'VAT code', 'Warehouse codeDescription', 'Receipt date']
Excel exported: export/clean_PO.xlsx


In [59]:
import pandas as pd
from IPython.display import display

path = "data/products_export_1.csv"


pd.set_option('display.max_columns', None)

df = pd.read_csv(
    path,
    encoding="utf-8",
    engine="python"
)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df_view = df[
    [
        "Title",
        "Variant SKU",
        "ETA (product.metafields.custom.eta)"
    ]
].rename(columns={
    "Variant SKU": "SKU",
    "ETA (product.metafields.custom.eta)": "ETA"
})

# delete rows where SKU is NaN
df_view = df_view.dropna(subset=["SKU"])

display(df_view.head(100))
print("Rows:", len(df_view))


# export to Excel
output_path = "export/clean_product_shopify.xlsx"
df_view.to_excel(output_path, index=False)

print("Excel exported:", output_path)

Rows: 13097
Columns: ['Handle', 'Title', 'Body (HTML)', 'Vendor', 'Product Category', 'Type', 'Tags', 'Published', 'Option1 Name', 'Option1 Value', 'Option1 Linked To', 'Option2 Name', 'Option2 Value', 'Option2 Linked To', 'Option3 Name', 'Option3 Value', 'Option3 Linked To', 'Variant SKU', 'Variant Grams', 'Variant Inventory Tracker', 'Variant Inventory Qty', 'Variant Inventory Policy', 'Variant Fulfillment Service', 'Variant Price', 'Variant Compare At Price', 'Variant Requires Shipping', 'Variant Taxable', 'Unit Price Total Measure', 'Unit Price Total Measure Unit', 'Unit Price Base Measure', 'Unit Price Base Measure Unit', 'Variant Barcode', 'Image Src', 'Image Position', 'Image Alt Text', 'Gift Card', 'SEO Title', 'SEO Description', 'Google Shopping / Google Product Category', 'Google Shopping / Gender', 'Google Shopping / Age Group', 'Google Shopping / MPN', 'Google Shopping / Condition', 'Google Shopping / Custom Product', 'Google Shopping / Custom Label 0', 'Google Shopping / C

,Title,SKU,ETA
5,The Shell Purse on Stand,BAKU009NBr,NaN
8,The Island Fuzzy Chair - Natural Brown,JAAM021NBr,NaN
9,The Suar Bar Stool - OUD MODEL,JAAM034N,NaN
10,The Rope 75M - Natural,JACOX015N-75M,NaN
11,The Lonjong Dining Table - Indoor - 180x85,JAKAY010N-150x85,NaN
...,...,...,...
213,The Dumpling Floor Lamp - Pendant - Natural - M,ASM-BAYU004N-M_FL-JUTE-001,NaN
214,The Kendi Pendant - Natural - M,ASM-BA171N-M_FL-W-001,NaN
215,The Island Console - Black,JAAM025B,NaN
216,The Island Rope One Seater - Natural White_40,JAAM033NW,NaN


Rows: 2680
Excel exported: export/clean_product_shopify.xlsx


In [60]:
import pandas as pd

# -----------------------
# LOAD DATA
# -----------------------

products = pd.read_excel("export/clean_product_shopify.xlsx")
po = pd.read_excel("export/clean_PO.xlsx")

# samakan nama kolom
po = po.rename(columns={"Item": "SKU"})

# convert tanggal
po["Receipt date"] = pd.to_datetime(po["Receipt date"], errors="coerce")

today = pd.Timestamp.today().normalize()

# hanya ambil receipt date >= hari ini
po = po[po["Receipt date"] >= today]

# -----------------------
# AMBIL ETA TERDEKAT PER SKU
# -----------------------

eta_per_sku = (
    po.sort_values("Receipt date")
      .groupby("SKU", as_index=False)
      .first()[["SKU", "Receipt date"]]
)

eta_per_sku = eta_per_sku.rename(columns={"Receipt date": "ETA"})

# -----------------------
# MATCH DENGAN PRODUCTS
# -----------------------

result = products.merge(
    eta_per_sku,
    on="SKU",
    how="inner"
)

# jika ada kolom ETA lama
if "ETA_x" in result.columns:
    result["ETA"] = result["ETA_y"]
    result = result.drop(columns=["ETA_x", "ETA_y"])

# -----------------------
# FORMAT TANGGAL
# -----------------------

result["ETA"] = pd.to_datetime(result["ETA"]).dt.strftime("%d/%m/%Y")

# -----------------------
# EXPORT
# -----------------------

result.to_excel("export/products_with_eta.xlsx", index=False)

print("Export selesai: export/products_with_eta.xlsx")
print("Total SKU match:", len(result))

Export selesai: export/products_with_eta.xlsx
Total SKU match: 511
